In [6]:
%pip install lightgbm xgboost scikit-learn numpy pandas scipy



[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import signal, stats
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb

print('All packages loaded.')

All packages loaded.


In [8]:
DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
    return dead

TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
TEST_EDA_DEAD  = eda_dead_subjects(testeda)

def compute_subject_baseline(sensor_df, val_col):
    out = {}
    for pid, grp in sensor_df.groupby('pid'):
        v = grp[val_col].dropna()
        out[pid] = (v.mean(), v.std() + 1e-8)
    return out

def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

print('Data loaded.')
print('Train arousal dist:', train_labels['arousal'].value_counts().sort_index().to_dict())
print('Train shape:', train_labels.shape, '| Test shape:', test_labels.shape)


Data loaded.
Train arousal dist: {1: 55, 2: 430, 3: 554, 4: 345, 5: 72}
Train shape: (1456, 4) | Test shape: (1496, 4)


In [9]:
def approx_entropy(vals, m=2, r=None):
    """Approximate entropy: measure of signal complexity."""
    if len(vals) < m + 1:
        return np.nan
    if r is None:
        r = 0.2 * np.std(vals)
    if r == 0:
        return np.nan
    def _maxdist(x_i, x_j):
        return max([abs(ua - va) for ua, va in zip(x_i, x_j)])
    def _phi(m):
        x = [[vals[j] for j in range(i, i + m - 1 + 1)] for i in range(len(vals) - m + 1)]
        C = [len([1 for x_j in x if _maxdist(x_i, x_j) <= r]) / (len(vals) - m + 1.0) for x_i in x]
        return (len(vals) - m + 1.0) ** (-1) * sum(np.log(C))
    return abs(_phi(m + 1) - _phi(m))

def spectral_power(vals, freq_bands=None):
    """Compute power in freq bands via simple FFT."""
    if len(vals) < 5:
        return {'spec_total': np.nan, 'spec_low': np.nan, 'spec_high': np.nan, 'spec_ratio': np.nan}
    vals = np.array(vals)
    vals = (vals - np.mean(vals)) / (np.std(vals) + 1e-8)
    fft = np.abs(np.fft.fft(vals)) ** 2
    psd = fft / len(fft)
    
    out = {}
    out['spec_total'] = np.sum(psd)
    out['spec_low']   = np.sum(psd[:len(psd)//3])
    out['spec_high']  = np.sum(psd[2*len(psd)//3:])
    out['spec_ratio'] = out['spec_high'] / (out['spec_low'] + 1e-8)
    return out

def cross_correlation_lag0(x, y):
    """Peak cross-correlation at lag 0 (synchrony). Handles different-length arrays."""
    if len(x) < 2 or len(y) < 2:
        return np.nan
    # Truncate to same length
    n = min(len(x), len(y))
    x = np.array(x[:n], dtype=float)
    y = np.array(y[:n], dtype=float)
    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan
    corr = np.corrcoef(x, y)[0, 1]
    return corr if not np.isnan(corr) else np.nan

def reactivity_metric(vals):
    """2nd derivative: how quickly signal accelerates."""
    if len(vals) < 3:
        return np.nan
    vals = np.array(vals, dtype=float)
    d1 = np.diff(vals)
    d2 = np.diff(d1)
    return np.mean(np.abs(d2)) if len(d2) > 0 else np.nan

def outlier_ratio(vals, threshold=2.0):
    """% of samples > threshold*std from local mean."""
    if len(vals) < 2:
        return np.nan
    vals = np.array(vals)
    m, s = np.mean(vals), np.std(vals)
    if s < 1e-8:
        return 0.0
    return np.mean(np.abs(vals - m) > threshold * s)

print('Feature helper functions ready.')

Feature helper functions ready.


In [ ]:
WINDOWS_MS  = [5000, 10000]
ROLL_WIN_MS = 30000

def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
    else:
        for s in ['mean','std','range','slope','p25','p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat

def nan_eda_features(feat, wl):
    for key in list(feat.keys()):
        if f'eda_{wl}' in key and key not in [f'eda_{wl}_valid', f'eda_{wl}_zero_ratio']:
            feat[key] = np.nan
    return feat

def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp,
                         sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid','timestamp'], inplace=True)

    records = []
    for idx, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        feat['bl_hr']   = bl_hr.get(pid,   (np.nan,1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan,1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan,1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan,1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan,1))[0]

        def get_win(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            # ===== HR =====
            hr_v = get_win(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan
            
            # NEW: Spectral
            hr_spec = spectral_power(hr_v)
            for k, v in hr_spec.items():
                feat[f'hr_{wl}_{k}'] = v
            
            # NEW: Entropy
            feat[f'hr_{wl}_entropy']   = approx_entropy(hr_v)
            
            # NEW: Reactivity
            feat[f'hr_{wl}_reactivity'] = reactivity_metric(hr_v)
            
            # NEW: Outlier ratio
            feat[f'hr_{wl}_outlier_ratio'] = outlier_ratio(hr_v)

            # ===== EDA =====
            eda_v = get_win(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
                feat[f'eda_{wl}_nz_frac'] = len(nz) / len(eda_v)
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, wl)
            else:
                for k in ['zero_ratio','valid','dev','nz_mean','nz_frac']:
                    feat[f'eda_{wl}_{k}'] = np.nan
                feat = nan_eda_features(feat, wl)
            
            # NEW: EDA Spectral
            eda_spec = spectral_power(eda_v)
            for k, v in eda_spec.items():
                feat[f'eda_{wl}_{k}'] = v if pid not in eda_dead_set else np.nan
            
            # NEW: EDA Entropy
            feat[f'eda_{wl}_entropy'] = approx_entropy(eda_v) if pid not in eda_dead_set else np.nan
            
            # NEW: EDA Reactivity
            feat[f'eda_{wl}_reactivity'] = reactivity_metric(eda_v) if pid not in eda_dead_set else np.nan
            
            # NEW: EDA Outlier ratio
            feat[f'eda_{wl}_outlier_ratio'] = outlier_ratio(eda_v) if pid not in eda_dead_set else np.nan

            # ===== TEMP =====
            temp_v = get_win(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev'] = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan
            
            # NEW: Temp entropy & reactivity
            feat[f'temp_{wl}_entropy'] = approx_entropy(temp_v)
            feat[f'temp_{wl}_reactivity'] = reactivity_metric(temp_v)

            # ===== ACC =====
            acc_v = get_win(acc_df, 'magnitude', hw)
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            if len(acc_v) >= 5:
                feat[f'acc_{wl}_mean']   = np.mean(acc_v)
                feat[f'acc_{wl}_std']    = np.std(acc_v)
                feat[f'acc_{wl}_energy'] = np.mean(acc_v**2)
                feat[f'acc_{wl}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','energy','dev']:
                    feat[f'acc_{wl}_{s}'] = np.nan

            # ===== BVP =====
            bvp_v = get_win(bvp_df, 'value', hw)
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v,75) - np.percentile(bvp_v,25)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
            else:
                for s in ['std','range','iqr','dev']:
                    feat[f'bvp_{wl}_{s}'] = np.nan
            
            # NEW: BVP entropy & outlier
            feat[f'bvp_{wl}_entropy'] = approx_entropy(bvp_v)
            feat[f'bvp_{wl}_outlier_ratio'] = outlier_ratio(bvp_v)

            # ===== EEG =====
            s_eeg = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts-hw) & (s_eeg.timestamp < ts+hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = win_eeg[col].mean()
                th = feat[f'eeg_theta_{wl}']
                la = feat[f'eeg_lowAlpha_{wl}']
                ha = feat[f'eeg_highAlpha_{wl}']
                lb = feat[f'eeg_lowBeta_{wl}']
                hb = feat[f'eeg_highBeta_{wl}']
                lg = feat[f'eeg_lowGamma_{wl}']
                de = feat[f'eeg_delta_{wl}']
                feat[f'eeg_theta_alpha_{wl}']  = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wl}']   = (lb + hb) / (la + ha + eps)
                feat[f'eeg_hbeta_lgamma_{wl}'] = hb / (lg + eps)
                feat[f'eeg_engage_{wl}']       = hb / (de + th + eps)
            else:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = np.nan
                for r in ['theta_alpha','beta_alpha','hbeta_lgamma','engage']:
                    feat[f'eeg_{r}_{wl}'] = np.nan

        # ===== NEW: Cross-signal sync (HR ↔ EDA) =====
        s_hr = hr_df[hr_df.pid == pid]
        s_eda = eda_df[eda_df.pid == pid]
        win_hr = s_hr[(s_hr.timestamp >= ts - 5000) & (s_hr.timestamp < ts + 5000)]['value'].values
        win_eda = s_eda[(s_eda.timestamp >= ts - 5000) & (s_eda.timestamp < ts + 5000)]['value'].values
        feat['sync_hr_eda'] = cross_correlation_lag0(win_hr, win_eda)

        # Rolling deviation (v18)
        for sensor, df_, col in [('hr', hr_df, 'value'), ('temp', temp_df, 'value')]:
            s = df_[df_.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)][col].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)][col].values
            if len(past) >= 2 and len(cur) >= 1:
                feat[f'{sensor}_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat[f'{sensor}_roll_dev'] = np.nan

        if pid not in eda_dead_set:
            s = eda_df[eda_df.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)]['value'].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)]['value'].values
            if len(past) >= 2 and len(cur) >= 1:
                feat['eda_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat['eda_roll_dev'] = np.nan
        else:
            feat['eda_roll_dev'] = np.nan

        # Interactions (v18 baseline)
        hr_m   = feat.get('hr_w5s_mean',   np.nan)
        hr_d   = feat.get('hr_w5s_dev',    np.nan)
        eda_d  = feat.get('eda_w5s_dev',   np.nan)
        tmp_m  = feat.get('temp_w5s_mean', np.nan)
        feat['hr_temp_product']    = hr_m * tmp_m
        feat['dev_hr_eda_product'] = hr_d * eda_d

        records.append(feat)
        
        if (idx + 1) % 300 == 0:
            print(f'  {idx + 1}/{len(label_df)} rows processed')

    return pd.DataFrame(records)


print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)

print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)

print(f'Train features: {train_feats.shape}')
print(f'Test features: {test_feats.shape}')

Extracting TRAIN features...


In [ ]:
LAG_BASE = ['hr_w5s_mean', 'hr_w5s_dev', 'hr_roll_dev',
            'eda_w5s_mean', 'eda_w5s_dev', 'eda_roll_dev',
            'temp_w5s_mean', 'temp_w5s_dev', 'temp_roll_dev',
            'bvp_w5s_std']
LAG_COLS = [c for c in LAG_BASE if c in train_feats.columns]

def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid','timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    for col in cols:
        df[f'{col}_roll3'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    return df

train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id','pid','timestamp','arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]

# Drop NaN-heavy features (>50% missing)
nan_counts = train_feats[FEAT_COLS].isnull().sum()
drop_cols = nan_counts[nan_counts > len(train_feats) * 0.5].index.tolist()
FEAT_COLS = [c for c in FEAT_COLS if c not in drop_cols]

print(f'Features after NaN-drop: {len(FEAT_COLS)} (dropped {len(drop_cols)})')
print(f'Top dropped (>50% NaN): {drop_cols[:5]}')


In [ ]:
train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_all)
TRAIN_PRIOR = np.bincount(y_all, minlength=5) / len(y_all)

print(f'Class weights: {dict((f"A{i+1}", round(w, 2)) for i, w in enumerate(cw))}')
print(f'Train prior: {dict((f"A{i+1}", round(p, 3)) for i, p in enumerate(TRAIN_PRIOR))}')


In [ ]:

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS      = [42, 7, 123, 13, 99]

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_xgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float64)

loso_lgb, loso_xgb = [], []

def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=63, learning_rate=0.03,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=15, lambda_l1=0.3, lambda_l2=0.3,
        max_depth=7, verbose=-1, seed=seed, n_jobs=-1
    )

def get_xgb_params(seed):
    return dict(
        objective='multi:softprob', num_class=5, eval_metric='mlogloss',
        max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7,
        min_child_weight=10, reg_alpha=0.3, reg_lambda=0.3,
        seed=seed, verbosity=0, nthread=-1
    )

for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    sw_tr = cw[y_tr]

    f_lgb = np.zeros((va_mask.sum(), 5))
    f_xgb = np.zeros((va_mask.sum(), 5))
    t_lgb = np.zeros((len(test_feats), 5))
    t_xgb = np.zeros((len(test_feats), 5))

    for seed in SEEDS:
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr, num_boost_round=1500,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        f_lgb += m.predict(X_va)   / len(SEEDS)
        t_lgb += m.predict(X_test) / len(SEEDS)

        dtr_x = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
        dva_x = xgb.DMatrix(X_va, label=y_va)
        m_x = xgb.train(
            get_xgb_params(seed), dtr_x, num_boost_round=1500,
            evals=[(dva_x, 'val')],
            early_stopping_rounds=100, verbose_eval=False,
        )
        f_xgb += m_x.predict(dva_x).reshape(-1,5)               / len(SEEDS)
        t_xgb += m_x.predict(xgb.DMatrix(X_test)).reshape(-1,5) / len(SEEDS)

    oof_lgb[va_mask] = f_lgb
    oof_xgb[va_mask] = f_xgb
    test_lgb += t_lgb / len(TRAIN_PIDS)
    test_xgb += t_xgb / len(TRAIN_PIDS)

    ba_l = balanced_accuracy_score(y_va, f_lgb.argmax(axis=1))
    ba_x = balanced_accuracy_score(y_va, f_xgb.argmax(axis=1))
    loso_lgb.append(ba_l)
    loso_xgb.append(ba_x)
    print(f'  {fold_pid} — LGB: {ba_l:.4f} | XGB: {ba_x:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_lgb):.4f} ± {np.std(loso_lgb):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb):.4f} ± {np.std(loso_xgb):.4f}')


In [ ]:
X_meta_train = np.hstack([oof_lgb, oof_xgb]).astype(np.float32)
X_meta_test  = np.hstack([test_lgb, test_xgb]).astype(np.float32)

meta_params = dict(
    objective='multi:softprob', num_class=5, eval_metric='mlogloss',
    max_depth=2, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    min_child_weight=1, reg_alpha=0.1, reg_lambda=0.1,
    seed=42, verbosity=0, nthread=-1
)
dtr_meta = xgb.DMatrix(X_meta_train, label=y_all, weight=cw[y_all])
meta_model = xgb.train(
    meta_params, dtr_meta, num_boost_round=500,
    callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True)]
)

oof_meta = meta_model.predict(xgb.DMatrix(X_meta_train))
test_meta = meta_model.predict(xgb.DMatrix(X_meta_test))

ba_meta = balanced_accuracy_score(y_all, oof_meta.argmax(axis=1))
print(f'\nStacking OOF BA: {ba_meta:.4f}')

In [ ]:
oof_calib = oof_meta.copy()

for c in range(5):
    mask_c = y_all == c
    if mask_c.sum() >= 10:
        iso = IsotonicRegression(out_of_bounds='clip')
        oof_calib[mask_c, c] = iso.fit_transform(
            oof_meta[mask_c, c],
            y_all[mask_c] == c
        ).astype(np.float64)

oof_calib = oof_calib / (oof_calib.sum(axis=1, keepdims=True) + 1e-8)

ba_calib = balanced_accuracy_score(y_all, oof_calib.argmax(axis=1))
print(f'After calibration OOF BA: {ba_calib:.4f}')

# Apply to test
test_calib = test_meta.copy()
for c in range(5):
    mask_c = y_all == c
    if mask_c.sum() >= 10:
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(oof_meta[mask_c, c], y_all[mask_c] == c)
        test_calib[:, c] = iso.predict(test_meta[:, c]).astype(np.float64)

test_calib = test_calib / (test_calib.sum(axis=1, keepdims=True) + 1e-8)


In [ ]:
test_pred = test_calib.argmax(axis=1) + 1
oof_pred  = oof_calib.argmax(axis=1)

print('\n=== Final OOF Classification Report ===')
print(classification_report(y_all, oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print(f'\nFinal OOF BA: {balanced_accuracy_score(y_all, oof_pred):.4f}')

print(f'\nTest distribution:')
test_dist = pd.Series(test_pred).value_counts().sort_index()
print(test_dist)

print(f'\nExpected from prior:')
for i, p in enumerate(TRAIN_PRIOR):
    print(f'  A{i+1}: {int(p * len(test_pred))}')

In [ ]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred,
})
submission.to_csv('submission-v24.csv', index=False)
print(f'\nsubmission-v24.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())
